In [ ]:
# Configure Kaggle credentials outside this notebook.
# Use ~/.kaggle/access_token locally or Kaggle Secrets in cloud notebooks.


# Notebook 1: QLoRA Fine-Tuning Pipeline for Text-to-SQL
### Target: `unsloth/Qwen2.5-Coder-7B-Instruct` on a single NVIDIA T4 GPU (15GB VRAM)

QLoRA fine-tuning of Qwen2.5-Coder-7B for text-to-SQL on Spider.

**Configuration:**
- **Quantization**: 4-bit NormalFloat4 (NF4) with double quantization via Unsloth kernels.
- **Target Model**: `Qwen2.5-Coder-7B-Instruct`.
- **LoRA Parameters**: Rank $r=16$, $\alpha=16$, targeting all 7 linear projection layers (`q, k, v, o, gate, up, down`).
- **Dataset**: `xlangai/spider` formatted using native chat templates with database IDs, questions, and SQL code fences.
- **Training Engine**: Hugging Face TRL (`SFTConfig` + `SFTTrainer`) with gradient accumulation and memory management tailored for 15GB VRAM limits.
- **Adapter Artifacts**: Save LoRA adapter weights separately from the base model.

## 1. Environment & Dependency Installation
Installing Unsloth, TRL, PEFT, BitsAndBytes, and Hugging Face ecosystem libraries.

In [ ]:
%%capture
# Install Unsloth and matching accelerated dependencies
!pip install --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-cache-dir trl peft bitsandbytes datasets transformers huggingface_hub accelerate

## 2. Hardware Diagnostics & Memory Verification
Verify CUDA device availability, check available VRAM (targeting 15GB T4), and prepare memory metrics.

In [ ]:
import torch
import os

assert torch.cuda.is_available(), "CUDA GPU not detected! Please attach a T4 GPU in Colab / Kaggle."

gpu_name = torch.cuda.get_device_name(0)
gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f"[GPU Init] Active Device: {gpu_name}")
print(f"[GPU Init] Total VRAM: {gpu_memory_gb:.2f} GB")

# Optimization flags
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

## 3. Base Model & QLoRA Configuration
Load `unsloth/Qwen2.5-Coder-7B-Instruct` in 4-bit NF4 precision. Attach LoRA adapters with $r=16, \alpha=16$ to all projection heads.

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 2048
BASE_MODEL_NAME = "unsloth/Qwen2.5-Coder-7B-Instruct"
LOAD_IN_4BIT = True

# 1. Load 4-bit base model & tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
    dtype=None,  # Auto-select float16 or bfloat16 based on hardware capability
)

# 2. Apply QLoRA parameter configuration
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_dropout=0.0,  # 0 is optimized by Unsloth
    bias="none",
    use_gradient_checkpointing="unsloth",  # Reduces VRAM consumption by ~60%
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

print("[Model Ready] Trainable parameters summary:")
model.print_trainable_parameters()

## 4. Dataset Ingestion & Chat Template Formatting
Load the `xlangai/spider` dataset and map each example into conversational messages using Qwen 2.5's native chat template format:
- **System**: `"You are an expert SQL assistant. Given a database schema and a question, write a valid SQL query that answers the question."`
- **User**: `"Database: {db_id}\nQuestion: {question}"`
- **Assistant**: `"```sql\n{query}\n```"`

In [ ]:
from datasets import load_dataset

# Load Spider benchmark dataset
raw_dataset = load_dataset("xlangai/spider", split="train")
print(f"[Dataset] Loaded {len(raw_dataset)} training samples from xlangai/spider")

SYSTEM_PROMPT = "You are an expert SQL assistant. Given a database schema and a question, write a valid SQL query that answers the question."

def format_spider_prompts(batch):
    """
    Formats raw Spider rows into Qwen conversational format applying the tokenizer's chat template.
    """
    formatted_texts = []
    for db_id, question, query in zip(batch["db_id"], batch["question"], batch["query"]):
        conversation = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Database: {db_id}\nQuestion: {question}"},
            {"role": "assistant", "content": f"```sql\n{query}\n```"},
        ]
        formatted_text = tokenizer.apply_chat_template(
            conversation,
            tokenize=False,
            add_generation_prompt=False,
        )
        formatted_texts.append(formatted_text)
    return {"text": formatted_texts}

# Map formatting across dataset
train_dataset = raw_dataset.map(
    format_spider_prompts,
    batched=True,
    remove_columns=raw_dataset.column_names,
    desc="Formatting Chat Templates",
)

print("\n--- Sample Formatted Conversation ---")
print(train_dataset[0]["text"]) if len(train_dataset) > 0 else None

## 5. SFT Training Pipeline (TRL `SFTTrainer` & `SFTConfig`)
Configure the training arguments to stay within a single T4's 15GB memory envelope:
- Per-device batch size: `2`
- Gradient accumulation steps: `4` (effective batch size = $2 \times 4 = 8$)
- Learning rate: `2e-4` with linear decay
- Max sequence length: `2048`
- Optimizer: `adamw_8bit`
- FP16 precision (T4 does not have native BF16 acceleration)

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

training_args = SFTConfig(
    output_dir="outputs_qlora_sql",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    max_steps=60,  # Demonstrative run; set to num_train_epochs=1 for full epoch
    learning_rate=2e-4,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=5,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=3407,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    report_to="none",  # Disable external loggers (wandb/mlflow) for headless environments
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,  # Can set to True for faster packing, False for strict boundary preservation
    args=training_args,
)

print("[Training] Launching SFTTrainer optimization loop...")
train_stats = trainer.train()
print(f"[Training Complete] Peak memory: {torch.cuda.max_memory_allocated() / (1024**3):.2f} GB")

## 6. Inference Sanity Check
Generate a sample query with the trained adapter.

In [ ]:
# Activate Unsloth native fast inference mode
FastLanguageModel.for_inference(model)

test_messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {
        "role": "user",
        "content": "Database: concert_singer\nQuestion: How many singers do we have?",
    },
]

inputs = tokenizer.apply_chat_template(
    test_messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=128,
    use_cache=True,
    temperature=0.1,
)

response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print("--- Model SQL Output ---")
print(response)

## 7. LoRA Adapter Export to Hugging Face Hub
Export the trained LoRA adapter separately from the base weights. The recorded FP32 adapter occupies approximately 161.5 MB.

In [ ]:
import huggingface_hub

# Set a Hugging Face access token with write permissions
HF_TOKEN = "hf_YOUR_TOKEN_HERE"
HF_REPO_NAME = "username/qwen2.5-coder-7b-text2sql-qlora"

# 1. Save local LoRA weights (~50MB)
LOCAL_ADAPTER_DIR = "lora_sql_adapter"
model.save_pretrained(LOCAL_ADAPTER_DIR)
tokenizer.save_pretrained(LOCAL_ADAPTER_DIR)
print(f"[Export] Saved local LoRA adapter to {LOCAL_ADAPTER_DIR}/")

# 2. Authenticate and push only adapters to Hugging Face Hub
if HF_TOKEN != "hf_YOUR_TOKEN_HERE" and HF_TOKEN.startswith("hf_"):
    huggingface_hub.login(token=HF_TOKEN)
    print(f"[Export] Pushing LoRA adapter to HF Hub: {HF_REPO_NAME}...")
    model.push_to_hub(HF_REPO_NAME, token=HF_TOKEN)
    tokenizer.push_to_hub(HF_REPO_NAME, token=HF_TOKEN)
    print("[Export Complete] LoRA adapter successfully published to Hugging Face Hub.")
else:
    print("[Notice] HF_TOKEN placeholder detected. Set HF_TOKEN to enable a Hub upload.")
    print(f"Local weights remain available in ./{LOCAL_ADAPTER_DIR}/")